In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
from llm_rag_faq import (
    AnthropicClient,
    LocalGGUFClient,
    RAGBase,
    build_index,
    load_faq_data,
)

In [ ]:
CACHE_DIR = "../.cache"
COURSE = "llm-zoomcamp"  # Or e.g. "mlops-zoomcamp", or None for ALL courses

documents = load_faq_data(course=None, cache_dir=CACHE_DIR)
index = build_index(documents, cache_dir=CACHE_DIR)

print(f"Indexed {len(index.documents)} documents (vector matrix: {index.doc_vectors.shape})")

In [4]:
# Release previous model first if re-running this cell to avoid OOM!
if "assistant" in globals():
    assistant.close()
if "local_client" in globals():
    local_client.close()

local_client = LocalGGUFClient(
    # Pinned Qwen2.5-1.5B-Instruct Q4_K_M by default; loads offline once cached
    n_ctx=4096,
    n_threads=2,            # Keep low (2-4) in small containers / Codespaces
    repeat_penalty=1.1,
    local_files_only=False, # Set True once downloaded to guarantee zero network calls
)

assistant = RAGBase(
    index=index,
    llm_client=local_client,
    course=COURSE,
    num_results=3,          # Smaller context keeps CPU inference fast
)

In [8]:
if "assistant" in globals():
    assistant.close()

api_key = os.getenv("ANTHROPIC_API_KEY")
assert api_key, "Set ANTHROPIC_API_KEY in .env"

client = Anthropic(api_key=api_key)
assistant = RAGBase(
    index=index,
    llm_client=client,
    course=None,
    model="claude-haiku-5-5",
)

In [5]:
question = "What is Telegram used for?"
answer = assistant.rag(question)
print(answer)

Telegram is used for announcements.
